In [3]:
import pandas as pd
import os
from pathlib import Path
import csv

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [12]:
dataset_specs = {
    "obis_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OBIS\obis_final.parquet"),
    },
    "gbif_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet"),
    },
    "ogsl_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OGSL\ogsl_final.parquet"),
    },
    "cb_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\CB\cb_final.parquet"),
    },
}

def normalize_type(value):
    t = str(value).strip().lower()
    # integers (including pandas nullable Int64 etc.)
    if any(x in t for x in ["int", "integer", "smallint", "bigint"]) or any(x in t for x in ["int64", "int32", "int16"]):
        return "integer"
    # floats (including pandas nullable Float64)
    if any(x in t for x in ["float", "double", "decimal", "numeric", "real"]) or any(x in t for x in ["float64", "float32"]):
        return "float"
    # datetimes
    if any(x in t for x in ["datetime", "timestamp", "date", "time"]) or "datetime64" in t:
        return "datetime"
    # booleans
    if "bool" in t or t == "boolean":
        return "boolean"
    # categorical types
    if any(x in t for x in ["category", "categorical"]):
        return "category"
    # strings: treat object, str, pandas string, pyarrow string, etc. as the same
    if any(x in t for x in ["string", "object", "char", "text", "varchar", "str", "pyarrow.string", "string[python]", "string[pyarrow]"]):
        return "string"
    # binary/bytes
    if "bytes" in t or "binary" in t:
        return "binary"
    return t

# Load all parquets
parquets = {}
schema_rows = []
comparison_rows = []

for dataset, spec in dataset_specs.items():
    parquet_path = spec["parquet"]
    if not parquet_path.exists():
        schema_rows.append({"dataset": dataset, "status": "missing parquet"})
        continue
    
    df = pd.read_parquet(parquet_path)
    parquets[dataset] = df
    schema_rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "status": "loaded"
    })

# Compare parquets to each other
datasets = list(parquets.keys())
all_columns = set()
for df in parquets.values():
    all_columns.update(df.columns)

for col in sorted(all_columns):
    for dataset1, dataset2 in [(datasets[i], datasets[j]) for i in range(len(datasets)) for j in range(i+1, len(datasets))]:
        df1 = parquets[dataset1]
        df2 = parquets[dataset2]
        
        in_df1 = col in df1.columns
        in_df2 = col in df2.columns
        
        if in_df1 and in_df2:
            type1 = normalize_type(str(df1[col].dtype))
            type2 = normalize_type(str(df2[col].dtype))
            if type1 != type2:
                comparison_rows.append({
                    "column": col,
                    "dataset1": dataset1,
                    "type1": str(df1[col].dtype),
                    "dataset2": dataset2,
                    "type2": str(df2[col].dtype),
                    "issue": "dtype mismatch"
                })
        elif in_df1 and not in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset1} only"
            })
        elif not in_df1 and in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset2} only"
            })

schema_report = pd.DataFrame(schema_rows)
comparison_report = pd.DataFrame(comparison_rows)

# Save to CSV
comparison_report.to_csv("parquet_comparison_report.csv", index=False)
schema_report.to_csv("parquet_schema_report.csv", index=False)

print(f"Saved comparison_report to: parquet_comparison_report.csv")
print(f"Saved schema_report to: parquet_schema_report.csv")

# Harmonise dtypes across dataframes and merge the four datasets
print("Harmonizing dtypes and merging datasets...")
# Determine target type for each column based on present types
target_types = {}
for col in sorted(all_columns):
    types = []
    for name, df in parquets.items():
        if col in df.columns:
            types.append(normalize_type(str(df[col].dtype)))
    if 'string' in types:
        target = 'string'
    elif 'datetime' in types:
        target = 'datetime'
    elif 'float' in types:
        target = 'float'
    elif 'integer' in types:
        target = 'integer'
    elif 'boolean' in types:
        target = 'boolean'
    elif 'category' in types:
        target = 'category'
    else:
        target = None
    target_types[col] = target

# Cast columns in each dataframe to the chosen target type (add missing columns)
for name, df in parquets.items():
    for col, target in target_types.items():
        if col not in df.columns:
            df[col] = pd.Series([pd.NA] * len(df), index=df.index)
        if target == 'string':
            df[col] = df[col].astype('string')
        elif target == 'datetime':
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif target == 'float':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
        elif target == 'integer':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif target == 'boolean':
            try:
                df[col] = df[col].astype('boolean')
            except Exception:
                df[col] = df[col].map({True: True, False: False}).astype('boolean')
        elif target == 'category':
            df[col] = df[col].astype('category')

# Concatenate in a consistent column order
col_order = sorted(all_columns)
merged = pd.concat([parquets[d] for d in datasets], ignore_index=True, sort=False)[col_order]

# Save merged dataset
merged_path = Path('All_data/merged_all.parquet')
merged.to_parquet(merged_path, index=False)
print(f"Saved merged to: {merged_path}")

schema_report, comparison_report, merged

Saved comparison_report to: parquet_comparison_report.csv
Saved schema_report to: parquet_schema_report.csv
Harmonizing dtypes and merging datasets...
Saved merged to: All_data\merged_all.parquet


(      dataset   n_rows  n_columns  status
 0  obis_final  1533558         61  loaded
 1  gbif_final  2588475         61  loaded
 2  ogsl_final     7257         61  loaded
 3    cb_final   243722         61  loaded,
 Empty DataFrame
 Columns: []
 Index: [],
            ID acceptedScientificName  aphiaID  areaSampled areaSampledUnit  \
 0        <NA>                  Colus   137704          NaN            <NA>   
 1        <NA>  Amphilectus digitatus   225695          NaN            <NA>   
 2        <NA>        Sarsia princeps   117487          NaN            <NA>   
 3        <NA>               Porifera      558          NaN            <NA>   
 4        <NA>  Ampharete octocirrata   332932          NaN            <NA>   
 ...       ...                    ...      ...          ...             ...   
 4373007  <NA>               Nemertea   152391         0.02              m2   
 4373008  <NA>               Nematoda      799         0.02              m2   
 4373009  <NA>               Bi

In [13]:
# Calculate coverage (non-null percentage) for each column
coverage_data = []
for col in col_order:
    total_rows = len(merged)
    non_null_count = merged[col].notna().sum()
    coverage_pct = (non_null_count / total_rows) * 100
    coverage_data.append({
        "column": col,
        "non_null_count": non_null_count,
        "total_rows": total_rows,
        "coverage_percent": coverage_pct
    })

coverage_report = pd.DataFrame(coverage_data)
coverage_report.to_csv("column_coverage.csv", index=False)
print(f"Saved column coverage report to: column_coverage.csv")
print(coverage_report)

Saved column coverage report to: column_coverage.csv
                    column  non_null_count  total_rows  coverage_percent
0                       ID               0     4373012          0.000000
1   acceptedScientificName         4372835     4373012         99.995952
2                  aphiaID         1784360     4373012         40.803913
3              areaSampled          223585     4373012          5.112838
4          areaSampledUnit          223585     4373012          5.112838
..                     ...             ...         ...               ...
56            verbatimDate         4249394     4373012         97.173161
57              verbatimID         4045497     4373012         92.510540
58  verbatimScientificName         4365087     4373012         99.818775
59               waterBody         3674045     4373012         84.016348
60                    year         4172438     4373012         95.413367

[61 rows x 4 columns]


In [17]:
# Create a source priority mapping
source_priority = {
    'CRITTERBASE': 1,
    'OGSL': 2,
    'OBIS': 3,
    'GBIF': 4
}

# Deduplication key columns
key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'acceptedScientificName']

# --- Preprocess / normalize fields so "missing" is consistent ---
# Normalize sourceDatabase (strip/case)
merged['sourceDatabase_norm'] = (
    merged['sourceDatabase'].astype('string')
          .str.strip()
          .str.upper()
)
merged['sourceDatabase_norm'] = merged['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Normalize numeric key columns (coerce invalid values to NaN)
for num_col in ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']:
    if num_col in merged.columns:
        merged[num_col] = pd.to_numeric(merged[num_col], errors='coerce')

# Normalize string key column
if 'acceptedScientificName' in merged.columns:
    merged['acceptedScientificName'] = (
        merged['acceptedScientificName'].astype('string')
              .str.strip()
              .replace({'': pd.NA, 'NA': pd.NA, 'N/A': pd.NA, 'NONE': pd.NA})
    )

# Identify rows with all key deduplication columns non-null (these are eligible for dedup)
mask = merged[key_cols].notna().all(axis=1)

# Split into rows to deduplicate and rows to keep as-is
to_dedup = merged[mask].copy()
to_keep = merged[~mask].copy()

# Add priority column based on normalized sourceDatabase
to_dedup['_priority'] = to_dedup['sourceDatabase_norm'].map(source_priority)

# Sort by key columns and priority (lower priority number = keep first); NaNs sort last
to_dedup_sorted = to_dedup.sort_values(by=key_cols + ['_priority'], na_position='last')

# Drop duplicates keeping first occurrence (highest priority)
to_dedup_deduped = to_dedup_sorted.drop_duplicates(subset=key_cols, keep='first')

# Remove the temporary priority column
to_dedup_deduped = to_dedup_deduped.drop(columns=['_priority'])

# Concatenate back together
merged_deduped = pd.concat([to_dedup_deduped, to_keep], ignore_index=True)

print(f"Original merged dataset: {len(merged)} rows")
print(f"After deduplication: {len(merged_deduped)} rows")
print(f"Rows deduplicated: {len(merged) - len(merged_deduped)}")

# Save the deduplicated dataset
merged_deduped.to_parquet('All_data/merged_all_deduplicated.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet")

# Compute which rows were removed during deduplication (from the to_dedup set)
removed = to_dedup_sorted.loc[~to_dedup_sorted.index.isin(to_dedup_deduped.index)].copy()
# Count removed rows by normalized sourceDatabase (include NaNs)
removed_counts = removed['sourceDatabase_norm'].value_counts(dropna=False).rename_axis('sourceDatabase').reset_index(name='removed_count')
print('Removed counts by sourceDatabase:')
print(removed_counts)
# Save the summary to CSV for further inspection
removed_counts.to_csv('dedup_removed_by_sourceDatabase.csv', index=False)
print('Saved removed counts to: dedup_removed_by_sourceDatabase.csv')

# Show a few example removed rows to inspect why higher-priority rows were kept
print('\nExamples of removed rows (head):')
print(removed.head(10))

# If you want, also save a small sample for manual inspection
removed.head(50).to_parquet('All_data/dedup_removed_sample.parquet', index=False)
print('Saved sample removed rows to: All_data/dedup_removed_sample.parquet')

Original merged dataset: 4373012 rows
After deduplication: 3361042 rows
Rows deduplicated: 1011970
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet
Removed counts by sourceDatabase:
  sourceDatabase  removed_count
0           OBIS         625057
1           GBIF         330280
2    CRITTERBASE          56630
3           OGSL              3
Saved removed counts to: dedup_removed_by_sourceDatabase.csv

Examples of removed rows (head):
           ID               acceptedScientificName  aphiaID  areaSampled  \
3912709  <NA>                           Tardigrada     <NA>          NaN   
3912770  <NA>                           Tardigrada     <NA>          NaN   
3913754  <NA>                           Tardigrada     <NA>          NaN   
3914730  <NA>                           Tardigrada     <NA>          NaN   
3916773  <NA>                           Tardigrada     <NA>          NaN   
318308   <NA>                    Polybius holsatus  1748246          NaN   
1285509 